---
# **Análise de Previsão de Demência**
---


🎯 **Objetivo específico:** Prever se o paciente apresenta sinais de demência.

🎯 **Objetivo geral:** Identificar as variáveis mais relevantes e propor uma análise baseada em modelos logísticos.

---


Desafio Estatística com Python - Regressão e Logística

Squad Nina da Hora | Bootcamp Data Analytics 2026.1

## 1. Configurações Iniciais

Variáveis da base de dados:

- `Subject ID`: Identificador único do paciente
- `MRI ID`: Identificador único do exame
- `Group`: Classificação do paciente
- `Visit`: Identificador da visita de cada paciente
- `MR Delay`: Intervalo em dias entre os exames
- `M/F`: Gênero (M: masculino, F: feminino)
- `Hand`: Mão dominante
- `Age`: Idade do paciente (numérico)
- `EDUC`: Anos de escolaridade (numérico)
- `SES`: Status socioeconômico (1 a 5)
- `MMSE`: Escore do Mini Exame do Estado Mental (0 a 30)
- `CDR`: Clinical Dementia Rating (0 a 3)
- `eTIV`: Volume intracraniano estimado
- `nWBV`: Proporção de volume cerebral normalizado
- `ASF`: Fator de escala anatômica

Variável alvo categórica: **Group**

● Nondemented - será tratada para variável binária 0 

● Converted - será tratada para variável binária 1

● Demented - será tratada para variável binária 1

In [1]:
# Importando as bibliotecas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample
from IPython.display import display, Markdown

In [2]:
# Configurações visuais dos gráficos

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
paleta = 'flare'
cores = sns.color_palette(paleta, n_colors=2) 

In [3]:
# Variáveis para reutilização

# data: dataframe contendo apenas as colunas de interesse
# nulos: colunas que possuem valores nulos
# var_features: seleção do df sem a variável alvo
var_alvo = 'Group'

In [4]:
# Lendo o dataframe

arquivo = 'oasis_longitudinal'
url = f'https://raw.githubusercontent.com/Squad-Nina-da-Hora/wmc-desafio-previsao-demencia/main/{arquivo}.csv'
df = pd.read_csv(url)

####  **Análise exploratória dos dados**

In [6]:
# Conhecendo os dados

print(f'\nTotal de linhas: {df.shape[0]}')
print(f'Total de colunas: {df.shape[1]}')
print('-' * 50)


Total de linhas: 373
Total de colunas: 15
--------------------------------------------------


In [7]:
# Verificando duplicatas

duplicados = df.duplicated().sum()
print(f'\nLinhas duplicadas na base: {duplicados}')


Linhas duplicadas na base: 0


In [8]:
# Verificando tipagem e nulos

info_df = pd.DataFrame({
    'Tipo': df.dtypes,
    'Valores Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df)) * 100,
    'Valores Únicos': df.nunique()
})
print('\n--- Diagnóstico de Tipagem e Qualidade ---\n')
display(info_df)


--- Diagnóstico de Tipagem e Qualidade ---



,Tipo,Valores Nulos,% Nulos,Valores Únicos
Subject ID,str,0,0.000000,150
MRI ID,str,0,0.000000,373
Group,str,0,0.000000,3
Visit,int64,0,0.000000,5
MR Delay,int64,0,0.000000,201
M/F,str,0,0.000000,2
Hand,str,0,0.000000,1
Age,int64,0,0.000000,39
EDUC,int64,0,0.000000,12
SES,float64,19,5.093834,5


In [9]:
# Armazenando colunas com valores nulos

nulos = df.columns[df.isna().any()].tolist()

print(f'Colunas com nulos identificadas: {nulos}')

Colunas com nulos identificadas: ['SES', 'MMSE']


In [10]:
# Conhecendo os dados

df.head()

,Subject ID,MRI ID,Group,Visit,MR Delay,M/F,Hand,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
0,OAS2_0001,OAS2_0001_MR1,Nondemented,1,0,M,R,87,14,2.0,27.0,0.0,1987,0.696,0.883
1,OAS2_0001,OAS2_0001_MR2,Nondemented,2,457,M,R,88,14,2.0,30.0,0.0,2004,0.681,0.876
2,OAS2_0002,OAS2_0002_MR1,Demented,1,0,M,R,75,12,NaN,23.0,0.5,1678,0.736,1.046
3,OAS2_0002,OAS2_0002_MR2,Demented,2,560,M,R,76,12,NaN,28.0,0.5,1738,0.713,1.010
4,OAS2_0002,OAS2_0002_MR3,Demented,3,1895,M,R,80,12,NaN,22.0,0.5,1698,0.701,1.034


In [11]:
# Conhecendo os dados

df.describe()

,Visit,MR Delay,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
count,373.000000,373.000000,373.000000,373.000000,354.000000,371.000000,373.000000,373.000000,373.000000,373.000000
mean,1.882038,595.104558,77.013405,14.597855,2.460452,27.342318,0.290885,1488.128686,0.729568,1.195461
std,0.922843,635.485118,7.640957,2.876339,1.134005,3.683244,0.374557,176.139286,0.037135,0.138092
min,1.000000,0.000000,60.000000,6.000000,1.000000,4.000000,0.000000,1106.000000,0.644000,0.876000
25%,1.000000,0.000000,71.000000,12.000000,2.000000,27.000000,0.000000,1357.000000,0.700000,1.099000
50%,2.000000,552.000000,77.000000,15.000000,2.000000,29.000000,0.000000,1470.000000,0.729000,1.194000
75%,2.000000,873.000000,82.000000,16.000000,3.000000,30.000000,0.500000,1597.000000,0.756000,1.293000
max,5.000000,2639.000000,98.000000,23.000000,5.000000,30.000000,2.000000,2004.000000,0.837000,1.587000


In [12]:
# Conhecendo os dados

df.describe(include=['object', 'string'])

,Subject ID,MRI ID,Group,M/F,Hand
count,373,373,373,373,373
unique,150,373,3,2,1
top,OAS2_0048,OAS2_0001_MR1,Nondemented,F,R
freq,5,1,190,213,373


In [13]:
# Conhecendo os dados

df['Group'].unique()

<StringArray>
['Nondemented', 'Demented', 'Converted']
Length: 3, dtype: str

In [14]:
# Conhecendo os dados

df['M/F'].unique()

<StringArray>
['M', 'F']
Length: 2, dtype: str

In [15]:
# Conhecendo os dados

df['Hand'].unique()

<StringArray>
['R']
Length: 1, dtype: str

In [16]:
# Selecionando apenas colunas de interesse

data = df.drop(['Subject ID', 'MRI ID', 'Visit', 'MR Delay', 'Hand'], axis=1)
data.head()

,Group,M/F,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
0,Nondemented,M,87,14,2.0,27.0,0.0,1987,0.696,0.883
1,Nondemented,M,88,14,2.0,30.0,0.0,2004,0.681,0.876
2,Demented,M,75,12,NaN,23.0,0.5,1678,0.736,1.046
3,Demented,M,76,12,NaN,28.0,0.5,1738,0.713,1.010
4,Demented,M,80,12,NaN,22.0,0.5,1698,0.701,1.034


In [17]:
# Variável categórica: % de group

data['Group'].value_counts(normalize=True)

Group
Nondemented    0.509383
Demented       0.391421
Converted      0.099196
Name: proportion, dtype: float64

In [18]:
# Variável categórica: tratamento de group

data['Group'] = data['Group'].map({'Demented': 1, 'Converted': 1, 'Nondemented': 0})
data.head()

,Group,M/F,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
0,0,M,87,14,2.0,27.0,0.0,1987,0.696,0.883
1,0,M,88,14,2.0,30.0,0.0,2004,0.681,0.876
2,1,M,75,12,NaN,23.0,0.5,1678,0.736,1.046
3,1,M,76,12,NaN,28.0,0.5,1738,0.713,1.010
4,1,M,80,12,NaN,22.0,0.5,1698,0.701,1.034


In [19]:
# Variável categórica: % de gênero

data['M/F'].value_counts(normalize=True)

M/F
F    0.571046
M    0.428954
Name: proportion, dtype: float64

In [20]:
# Variável categórica: tratamento de gênero

data['M/F'] = data['M/F'].map({'M': 1, 'F': 0})
data.head()

,Group,M/F,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
0,0,1,87,14,2.0,27.0,0.0,1987,0.696,0.883
1,0,1,88,14,2.0,30.0,0.0,2004,0.681,0.876
2,1,1,75,12,NaN,23.0,0.5,1678,0.736,1.046
3,1,1,76,12,NaN,28.0,0.5,1738,0.713,1.010
4,1,1,80,12,NaN,22.0,0.5,1698,0.701,1.034


In [21]:
# Fazendo a seleção do df sem a variável alvo (para reutilização)

var_features = [col for col in data.columns if col != var_alvo]
var_features

['M/F', 'Age', 'EDUC', 'SES', 'MMSE', 'CDR', 'eTIV', 'nWBV', 'ASF']